In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

url = '../data/raw/diamonds.csv'

data = pd.read_csv(url, sep=',')

data = data[(data['x'] > 0) & (data['y'] > 0) & (data['z'] > 0)]

cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

cut_mapping = {cat: i for i, cat in enumerate(cut_order)}
color_mapping = {cat: i for i, cat in enumerate(color_order)}
clarity_mapping = {cat: i for i, cat in enumerate(clarity_order)}

data['cut_encoded'] = data['cut'].map(cut_mapping)
data['color_encoded'] = data['color'].map(color_mapping)
data['clarity_encoded'] = data['clarity'].map(clarity_mapping)

features = ['carat', 'depth', 'table', 'x', 'y', 'z', 
            'cut_encoded', 'color_encoded', 'clarity_encoded']

X = data[features]
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"MAPE: {mape:.2f}%")

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
print(f"CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")



trained_model = pipeline.named_steps['model'] 

coef_df = pd.DataFrame({
    'Признак': features,
    'Коэффициент': trained_model.coef_
}).sort_values('Коэффициент', ascending=False)

print("\nКоэффициенты модели (после масштабирования):")
print(coef_df.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))


R²: 0.9100
RMSE: 1201.39
MAE: 790.37
MAPE: 42.99%
CV R²: 0.8875 (+/- 0.0355)

Коэффициенты модели (после масштабирования):
        Признак  Коэффициент
          carat   +5119.2634
clarity_encoded    +823.9814
  color_encoded    +550.0614
              y    +153.9515
    cut_encoded    +143.0037
          depth     -22.6375
          table     -55.3208
              x    -387.4342
              z    -757.2520
